# Sprint 10 · Gate sobre predições de validação · colab

Leia o `tutorial.md` da pasta. Input: sprint10_val.csv, com predições reais de validação. Não use o teste final nem cole tokens no notebook. O código já está embutido.

In [ ]:
from pathlib import Path

SMOKE = True  # S07–S09: teste curto; False usa o protocolo experimental
ARM = 'batch-16-fixed'  # usado apenas na S09
from google.colab import drive

drive.mount('/content/drive')
OUTPUT = Path('/content/drive/MyDrive/mammo')
DATA = OUTPUT / 'desenvolvimento'  # também aceita os CSVs diretamente em mammo/
if (OUTPUT / 'sprint10_val.csv').is_file():
    DATA = OUTPUT


In [ ]:
# ponytail: código e TOML embutidos; extração temporária, sem ZIP de código separado.
import subprocess, sys
from base64 import b64decode
from io import BytesIO
from pathlib import Path
from tempfile import TemporaryDirectory
from zipfile import ZipFile

CODE_DIR = TemporaryDirectory(prefix='mammo-codigo-')
PROJECT = Path(CODE_DIR.name).resolve()
with ZipFile(BytesIO(b64decode(
    '...'
    'AAAA'
))) as source:
    for name in source.namelist():
        if not (PROJECT / name).resolve().is_relative_to(PROJECT):
            raise ValueError('Caminho inválido no código embutido')
    source.extractall(PROJECT)
sys.path.insert(0, str(PROJECT / 'src'))


In [ ]:
from datetime import datetime, timezone
from tempfile import TemporaryDirectory
from zipfile import ZIP_DEFLATED, ZipFile

from project_cnn_mammo.protocols import run_sprint

stamp = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-%f')
archive = OUTPUT / f'sprint-10-{stamp}-resultados.zip'
with TemporaryDirectory(prefix='mammo-run-') as temp:
    out = Path(temp)
    try:
        print(run_sprint(10, DATA, out, smoke=False, arm=ARM, manifest_path=globals().get('MANIFEST')))
    finally:
        OUTPUT.mkdir(parents=True, exist_ok=True)
        pending = archive.with_suffix('.zip.part')
        with ZipFile(pending, 'w', ZIP_DEFLATED) as stream:
            for path in sorted(out.rglob('*')):
                if path.is_file():
                    stream.write(path, path.relative_to(out))
        with ZipFile(pending) as stream:
            if stream.testzip() is not None:
                raise RuntimeError(f'ZIP corrompido: {pending}')
        pending.replace(archive)
        print('ZIP:', archive)
        from google.colab import files
        files.download(str(archive))